In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
f_center = 436.5e6
f_bounds = (435e6, 438e6)
from scipy.constants import c

In [8]:
dielectric_materials = pd.DataFrame({
    "Material": ["Air", "Vacuum", "Teflon (PTFE)", "PLA", "PET-G", "ABS", "FR4"],
    "Dielectric Constant (εr)": [1.0006, 1.0, 2.1, 2.7, 3.0, 2.9, 4.5],
    "Loss Tangent (tan δ)": [0.0001, 0.0, 0.0002, 0.02, 0.02, 0.01, 0.02],
})
dielectric_materials

,Material,Dielectric Constant (εr),Loss Tangent (tan δ)
0,Air,1.0006,0.0001
1,Vacuum,1.0000,0.0000
2,Teflon (PTFE),2.1000,0.0002
3,PLA,2.7000,0.0200
4,PET-G,3.0000,0.0200
5,ABS,2.9000,0.0100
6,FR4,4.5000,0.0200


t - conductor height

h - dielectric height

W - width of the patch

L - length of the patch

eps_r - relative permittivity of the dielectric

eps_reff - effective relative permittivity of the dielectric

In [10]:
def eps_r_eff(eps_r, h, W):
  if W / h <= 1:
    raise ValueError("W/h must be greater than 1 for this formula to be valid.")
  return (eps_r + 1)/2 + (eps_r - 1)/2 * (1 + 12 * h/W)**(-0.5)

def L_fring(eps_reff, h, W):
  return h * 0.412 * (eps_reff + 0.3) * (W/h + 0.264) / ((eps_reff - 0.258) * (W/h + 0.8))

def f_r010(L, eps_reff):
  return c / (2 * L * np.sqrt(eps_reff))

def f_rc010(L, eps_reff, h, W):
  return c / (2 * (L + 2 * L_fring(eps_reff, h, W)) * np.sqrt(eps_reff))

def patch_width(eps_r, f0):
  return c / (2 * f0) * np.sqrt(2 / (eps_r + 1))

def patch_length(eps_reff, f0, delta_L):
  L = c / (2 * f0 * np.sqrt(eps_reff))
  return L - 2 * delta_L

In [14]:
# Example 14.1
# RT/duroid 5880, εr = 2.2, h = 1.57 mm, f0 = 10 GHz
W = patch_width(2.2, 10e9)
eps_reff = eps_r_eff(2.2, 1.57e-3, W)
delta_l = L_fring(eps_reff, 1.57e-3, W)
L = patch_length(eps_reff, 10e9, delta_l)

print(f"Patch Width (W): {W*1e3:.2f} mm")
print(f"Effective Dielectric Constant (εreff): {eps_reff:.4f}")
print(f"Fringing Length (ΔL): {delta_l*1e3:.4f} mm")
print(f"Patch Length (L): {L*1e3:.2f} mm")
print(f"Effective Patch Length (L_eff): {(L + 2 * delta_l)*1e3:.2f} mm")
print(f"Resonant Frequency (f_r010): {f_r010(L, eps_reff)/1e9:.4f} GHz")
print(f"Resonant Frequency with Fringing (f_rc010): {f_rc010(L, eps_reff, 1.57e-3, W)/1e9:.4f} GHz")

Patch Width (W): 11.85 mm
Effective Dielectric Constant (εreff): 1.9728
Fringing Length (ΔL): 0.8023 mm
Patch Length (L): 9.07 mm
Effective Patch Length (L_eff): 10.67 mm
Resonant Frequency (f_r010): 11.7696 GHz
Resonant Frequency with Fringing (f_rc010): 10.0000 GHz


In [ ]:
def patch_admittance(wavelength, W, h):
  k0 = 2 * np.pi / wavelength
  G = (W / (120 * wavelength)) * (1 - (1 / 24) * (k0 * h)**2)
  B = (W / (120 * wavelength)) * (1 - .636 * np.log(k0 * h))
  return complex(G, B)